# GBDT Stock Agent (Colab GPU)\nローカル検証後にこのノートブックを使って段階実行します。

In [ ]:
# 1) Drive mount and repo setup\nfrom google.colab import drive\ndrive.mount('/content/drive')\n\n!cd /content && rm -rf gbdt-stock-agent && git clone https://github.com/shonpyama/gbdt-stock-agent.git\n%cd /content/gbdt-stock-agent\n!python -m pip install --upgrade pip\n!pip install -e .

In [ ]:
# 2) API key and local-first persistence\nimport os\nfrom getpass import getpass\nfrom pathlib import Path\nfrom gbdt_agent.colab import setup_fast_colab_persistence\n\nif not os.environ.get('FMP_API_KEY'):\n    os.environ['FMP_API_KEY'] = getpass('FMP_API_KEY: ')\n\ndrive_path = Path('/content/drive/MyDrive/gbdt-stock-agent')\nsync_handle = setup_fast_colab_persistence(drive_path=drive_path, interval_seconds=600)\nprint('sync started:', drive_path)

In [ ]:
# 3) Preflight\n!python -m gbdt_agent.cli preflight --conf conf/default.yaml

In [ ]:
# 4) Stage controller\nfrom pathlib import Path\nfrom gbdt_agent.orchestrator import run_pipeline\n\nPROJECT_DIR = Path('/content/gbdt-stock-agent')\nCONF = PROJECT_DIR / 'conf/default.yaml'\n\ndef run_to(stage: str, force_unlock: bool = False):\n    run_id = run_pipeline(\n        project_dir=PROJECT_DIR,\n        conf_path=CONF,\n        resume=True,\n        force_unlock=force_unlock,\n        stop_after_stage=stage,\n    )\n    print('run_id=', run_id)\n    return run_id

In [ ]:
# 5) Execute by stage\nrun_id = run_to('stage_10_data_ready')\nrun_id = run_to('stage_20_validation_passed')\nrun_id = run_to('stage_30_features_ready')\nrun_id = run_to('stage_40_split_leakcheck_passed')\nrun_id = run_to('stage_50_models_trained')\nrun_id = run_to('stage_60_predictions_ready')\nrun_id = run_to('stage_70_backtest_ready')\nrun_id = run_to('stage_80_report_ready')

In [ ]:
# 6) Mandatory transition report (before move/hand-off)\n!python -m gbdt_agent.cli transition-report --run-id $run_id --target colab